# Demo del orquestador multi-agente

Este notebook demuestra el **flujo de delegación** de la pre-entrega 6: quién decide, a
quién le pasa la tarea, con qué instrucción y con qué aporte vuelve.

La pregunta está armada para que necesite **los dos dominios**: el dato (cuántos días de
vacaciones por tramo de antigüedad) lo tiene el **investigador** en las políticas internas,
y la cuenta total la hace el **analista** con la calculadora.

El orden lo fija el **supervisor**; los especialistas no se conocen entre sí.


## 1. La topología del grafo

Se arma una sola vez y se inspecciona. Los especialistas siempre vuelven al supervisor (ese
es el ciclo), y el nodo de `validacion` es el que decide si ya se puede cerrar.


In [1]:
from graph import build_app, diagrama_mermaid

app = build_app()
print(diagrama_mermaid(app))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	validacion(validacion)
	sintesis(sintesis)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analista --> supervisor;
	investigador --> supervisor;
	supervisor -.-> analista;
	supervisor -.-> investigador;
	supervisor -.-> validacion;
	validacion -.-> sintesis;
	validacion -.-> supervisor;
	sintesis --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. La corrida completa

`correr()` recorre el grafo y va mostrando cada paso: la decisión del supervisor (con la
instrucción que le deja al especialista), el aporte que devuelve cada agente y el informe
del validador. Al final imprime la respuesta para el usuario.


In [2]:
from main import correr

resultado, traza = correr()

PREGUNTA: Un colaborador cumplió 12 años en la empresa: 2 años con menos de 3 de antigüedad, 6 años con entre 3 y 8, y 4 años con más de 8. ¿Cuántos días de vacaciones le correspondieron en total a lo largo de esos 12 años?

FLUJO DE DELEGACIÓN
----------------------------------------------------------------------------------------


🔹 supervisor    decide -> investigador | instrucción: Buscá en las políticas internas de vacaciones cuántos días le corresponden a un colaborado


🔹 investigador  [investigador] Según la política interna:

- **Menos de 3 años de antigüedad:** 15 días hábiles de vacaciones por año calendario.
- **Entre 3 y 8 años de antigüedad (3 años o más, y hasta 8 años):** 20 días hábiles.
- **Más de 8 años d


🔹 supervisor    decide -> analista | instrucción: Calculá con los números que ya están en el contexto la cuenta que responde la pregunta ori


🔹 analista      [analista] Resultado: 250
Cuenta realizada: 2*15 + 6*20 + 4*25


🔹 supervisor    decide -> investigador | instrucción: Buscá en la política interna de vacaciones cuántos días hábiles le corresponden por año a 


🔹 investigador  [investigador] «Los colaboradores con más de 8 años de antigüedad acceden a 25 días hábiles.»

[Fuente: politica_vacaciones.txt]


🔹 supervisor    decide -> FINISH
🔹 validacion    suficiente=True | Los dos dominios aportaron y ninguno devolvió error: se puede sintetizar.


🔹 sintesis      [supervisor] A lo largo de los 12 años, al colaborador le correspondieron en total **250 días hábiles** de vacaciones. 

El desglose según la política interna es el siguiente:

* **Primeros 2 años** (menos de 3 años de antigüedad): 2

----------------------------------------------------------------------------------------
RESPUESTA FINAL

A lo largo de los 12 años, al colaborador le correspondieron en total **250 días hábiles** de vacaciones. 

El desglose según la política interna es el siguiente:

* **Primeros 2 años** (menos de 3 años de antigüedad): 2 años × 15 días = **30 días hábiles**
* **Siguientes 6 años** (entre 3 y 8 años de antigüedad): 6 años × 20 días = **120 días hábiles**
* **Últimos 4 años** (más de 8 años de antigüedad): 4 años × 25 días = **100 días hábiles**

**Total:** 30 + 120 + 100 = **250 días hábiles**.
----------------------------------------------------------------------------------------
pasos de especialistas: 3 | contribuciones: 3 | rúbrica

## 3. El estado compartido: quién aportó qué

Esto es lo que evita la pérdida de contexto. Cada contribución queda registrada con su
autor, y el supervisor decide mirando esa lista.


In [3]:
print('RESPUESTA FINAL:\n')
print(traza['respuesta_final'])

print('\n' + '=' * 70)
print('CONTRIBUCIONES (quién aportó qué)')
print('=' * 70)
for c in traza['contribuciones']:
    print(f"\n[{c['agente']}]\n{c['aporte']}")

print('\n' + '=' * 70)
print('INFORME DEL VALIDADOR')
print('=' * 70)
print(traza['validacion'])
print(f"\nPasos de especialistas: {traza['pasos_ejecutados']}")

RESPUESTA FINAL:

A lo largo de los 12 años, al colaborador le correspondieron en total **250 días hábiles** de vacaciones. 

El desglose según la política interna es el siguiente:

* **Primeros 2 años** (menos de 3 años de antigüedad): 2 años × 15 días = **30 días hábiles**
* **Siguientes 6 años** (entre 3 y 8 años de antigüedad): 6 años × 20 días = **120 días hábiles**
* **Últimos 4 años** (más de 8 años de antigüedad): 4 años × 25 días = **100 días hábiles**

**Total:** 30 + 120 + 100 = **250 días hábiles**.

CONTRIBUCIONES (quién aportó qué)

[investigador]
Según la política interna:

- **Menos de 3 años de antigüedad:** 15 días hábiles de vacaciones por año calendario.
- **Entre 3 y 8 años de antigüedad (3 años o más, y hasta 8 años):** 20 días hábiles.
- **Más de 8 años de antigüedad:** 25 días hábiles.

[Fuente: politica_vacaciones.txt]

[analista]
Resultado: 250
Cuenta realizada: 2*15 + 6*20 + 4*25

[investigador]
«Los colaboradores con más de 8 años de antigüedad acceden a 25 

## 4. Qué demuestra esto

- **Topología jerárquica**: un supervisor que rutea y decide el final; dos especialistas con
  herramientas acotadas (búsqueda y cálculo) que no se llaman entre ellos.
- **Estado compartido estructurado**: `contribuciones` con reducer `operator.add`, más
  `next_agent`, `pasos`, `task_completed` y el informe de `validacion`.
- **Flujo de supervisión**: el supervisor delega, recibe el aporte y decide si falta algo.
- **Corte del "Supervisor Infinito"**: `MAX_PASOS` en el supervisor, el validador
  determinista y `MAX_REINTENTOS` en las aristas.
- **Contexto acotado**: a cada especialista le llega una instrucción puntual, la pregunta
  original y los aportes (no el historial entero).
